# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Token loaded and login successful!")

Token loaded and login successful!


In [5]:
from huggingface_hub import HfFileSystem

fs = HfFileSystem()
files = fs.ls("datasets/FlyRank/internship-warehouse", detail=False)
for f in files:
    print(f)



files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance", detail=False)
for f in files:
    print(f)

datasets/FlyRank/internship-warehouse/fact_content_daily_performance
datasets/FlyRank/internship-warehouse/.gitattributes
datasets/FlyRank/internship-warehouse/README.md
datasets/FlyRank/internship-warehouse/dim_clients.parquet
datasets/FlyRank/internship-warehouse/dim_content.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet
datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05
datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06
datasets/FlyRank/internship-warehouse/fact_content_daily_perfor

In [6]:
files = fs.ls("datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03", detail=False)
for f in files:
    print(f)

datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


In [7]:
import pandas as pd

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df_march = pd.read_parquet(url)

print("Shape:", df_march.shape)
print("Date range:", df_march["report_date"].min(), "to", df_march["report_date"].max())
df_march.head()

Shape: (9841378, 31)
Date range: 2026-03-01 to 2026-03-31


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In fact_content_daily_performance, one row represents one page, for one client, on one specific day — not a single overall snapshot like in the starter CSV. I proved this by checking one page (content_id: content_b7e512995f79d5a6) for the client client_73cda7b4e4f265ea across March 2026: it had exactly 31 rows, one for each day of the month. This confirms the table works like a daily diary — every page gets a fresh entry each day it's tracked, rather than one lifetime total. This matters for my lane because any feature I build (like average CTR or engagement) needs to be calculated across these daily rows for a page, not read off a single row.

In [8]:
# Pick ONE page and see how many rows it has in March
sample_content = df_march["content_hash_id"].iloc[0]
sample_client = df_march["client_hash_id"].iloc[0]

rows_for_this_page = df_march[
    (df_march["content_hash_id"] == sample_content) &
    (df_march["client_hash_id"] == sample_client)
]

print("Page ID:", sample_content)
print("Number of rows for this one page in March:", rows_for_this_page.shape[0])
rows_for_this_page[["report_date", "client_hash_id", "content_hash_id"]]

Page ID: content_b7e512995f79d5a6
Number of rows for this one page in March: 31


,report_date,client_hash_id,content_hash_id
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
555866,2026-03-03,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1334822,2026-03-04,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1499377,2026-03-05,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1573240,2026-03-07,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
1594884,2026-03-02,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2009723,2026-03-08,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2281373,2026-03-06,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2439321,2026-03-09,client_73cda7b4e4f265ea,content_b7e512995f79d5a6
2553195,2026-03-10,client_73cda7b4e4f265ea,content_b7e512995f79d5a6


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.